# IND320 project 2026
Philip A. K. Øvland \
Github:     https://github.com/Povland/IND320 \
Streamlit:  https://ind320-2026.streamlit.app/

Video explaining stuff (Please tell me if the link doesn't work): TO DO

## AI Usage

I used AI as a helping hand with:
    - planning
    - bug fixing
    - first version of code blocks
    - making sure I had the correct setup locally
    - giving suggestions to the log, this part, readme and structure.

Getting AI to help with logging (not doing it blindly), helped me remember all the problems I came across.


## Log - Tasks
### Post oblig2 work: To make stuff look nicer
- Renamed and reordered the pages with `st.navigation`, wide layout, version in the sidebar (from `pyproject.toml`).
- Home: connection test on load and with a button, data status box, refresh button.
- New About page. Its text is shared with the info boxes on the other pages.
- Cross-border flow: choose connections by country or one by one, several years and months, direction filter, separate views for pie and plot, full-width plot, titles built from the selection.
- Cleanup: comments, imports, README, `.gitignore`, removed duplicates.

### Accounts and repositories
- They are obviously reused
- On a separate branch (byggeplass)

### API
- Familirazed



# Setup
Imports used throughout the notebook. API access lives in the `ind320` package (`src/ind320/`), and static files (shapefiles, CSV) are read from `data/` (described in `data/data_info`). Everything, including geopandas, is installed with `uv sync`.

In [ ]:
import numpy as np
import geopandas as gpd
import shapely

DATA_PATH = Path("../data")

# geopandas reads the zip files directly, no need to unzip.
# NVE_Region has no encoding file, so utf-8 is set by hand (otherwise "RØ" turns into "RÃ\x98")
price_areas = gpd.read_file(DATA_PATH / "Budområder.zip")
nve_regions = gpd.read_file(DATA_PATH / "NVE_Region.zip", encoding="utf-8")

print("CRS:", price_areas.crs, "and", nve_regions.crs)
print("Price areas:", list(price_areas["budomr"]))
print("NVE regions:", list(nve_regions["REGION"]))

### Shapefiles from NVE Atlas
Downloaded from https://atlas.nve.no on 09.11.2026 (see `data/data_info`):
- `Budområder.zip` (Nettanlegg > Nettområder > Budområder): the price areas NO1–NO5, matching `area_type = EL`
- `NVE_Region.zip` (Administrasjon > Administrative områder > NVE_Region): NVE's five regions, merged into the watercourse areas `area_type = VASS`

| VASS area_nr | Area | NVE regions |
|---|---|---|
| 1 | East | RS + RØ |
| 2 | West | RV |
| 3 | North | RM + RN |

NVE's area list describes VASS 3 as Midt-Norge down to southern Nordland, with VASS 4 for the rest of the north. But VASS 4 has no data (see Oblig2), and NVE's weekly report says area 3 covers Nord-Norge too, so RN goes into VASS 3, as the task says.

Both layers follow watersheds, so the NVE regions are close to the price areas (RØ ≈ NO1, RS ≈ NO2, RV ≈ NO5, RM ≈ NO3, RN ≈ NO4). The main difference is in the north: RM goes up to Saltfjellet, while NO3 stops at Tunnsjødal.

### Same area numbers as the reservoir data
Both layers get the columns `area_type` and `area_nr`, so they can be joined with the reservoir data later.

In [ ]:
# Price areas: "NO 1" -> area_nr 1
price_areas["area_type"] = "EL"
price_areas["area_nr"] = price_areas["budomr"].str.replace("NO", "").str.strip().astype(int)
price_areas = price_areas[["area_type", "area_nr", "geometry"]]

# Watercourse areas: the five NVE regions merged into the three VASS areas
REGION_TO_VASS = {"RS": 1, "RØ": 1, "RV": 2, "RM": 3, "RN": 3}

nve_regions["area_nr"] = nve_regions["REGION"].map(REGION_TO_VASS)
vass_areas = nve_regions.dissolve(by="area_nr").reset_index()
vass_areas["area_type"] = "VASS"
vass_areas = vass_areas[["area_type", "area_nr", "geometry"]]

# All areas in one table, 5 EL + 3 VASS = 8 rows
areas = pd.concat([price_areas, vass_areas], ignore_index=True)
areas

### Clipping to land
Both layers go out into the sea along the coast. Points at sea give weather data that doesn't belong to anywhere relevant, so the areas are clipped to the land of Norway first.

In [ ]:
# Land of Norway from NVE Atlas (Administrasjon > Administrative områder > Norge_N5000).
# If the file is missing, the areas are used as they are, sea included
LAND_PATH = DATA_PATH / "Norge_N5000.zip"

if LAND_PATH.exists():
    norway = gpd.read_file(LAND_PATH).to_crs(areas.crs)
    areas = gpd.clip(areas, norway)
    print("Clipped to land")
else:
    print(f"{LAND_PATH} not found, areas include the sea")